In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

: 

In [ ]:
file_path = 'parkinsons.csv'
df = pd.read_csv(file_path)

In [ ]:
print(df.head())

In [ ]:
print(df.describe())

In [ ]:
print(df.info())

In [ ]:
sns.countplot(x='status', data=df)
plt.title('Distribution of Parkinson\'s Disease Status')
plt.show()

In [ ]:
selected_features = ['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Shimmer(dB)']
plt.figure(figsize=(16, 8))
for i, feature in enumerate(selected_features, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(x='status', y=feature, data=df)
    plt.title(f'Boxplot of {feature}')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 8))
for i, feature in enumerate(selected_features, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df[feature], kde=True)
    plt.title(f'Distribution of {feature}')

plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(x='MDVP:Jitter(%)', y='MDVP:Shimmer(dB)', hue='status', data=df)
plt.title('Scatter Plot of Jitter vs Shimmer')
plt.show()

In [ ]:
plt.figure(figsize=(16, 8))
for i, feature in enumerate(selected_features, 1):
    plt.subplot(2, 3, i)
    sns.violinplot(x='status', y=feature, data=df)
    plt.title(f'Violin Plot of {feature}')

plt.tight_layout()
plt.show()

In [ ]:
df_numeric = df.select_dtypes(include=[float, int])

correlation_matrix = df_numeric.corr()
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()


Pre Processing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = df.drop('name', axis=1)


X = df.drop('status', axis=1)
y = df['status']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

In [ ]:
print("X_train_scaled:")
print(X_train_scaled_df.head())

In [ ]:
print("\nX_test_scaled:")
print(X_test_scaled_df.head())

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
X_train_scaled = imputer.fit_transform(X_train_scaled)
X_test_scaled = imputer.transform(X_test_scaled)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Missing values in X_train_scaled:")
print(X_train_scaled_df.isnull().sum())

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

X_train_pca_df = pd.DataFrame(X_train_pca, columns=[f'PC_{i+1}' for i in range(X_train_pca.shape[1])])
X_test_pca_df = pd.DataFrame(X_test_pca, columns=[f'PC_{i+1}' for i in range(X_test_pca.shape[1])])

print("\nX_train_pca:")
print(X_train_pca_df.head())

print("\nX_test_pca:")
print(X_test_pca_df.head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler
from keras.models import Sequential
from keras.layers import LSTM, GRU, Dense, Bidirectional
from keras.optimizers import Adam


df = pd.read_csv('/content/parkinsons.csv')


X = df.drop(['name', 'status'], axis=1).values
y = df['status'].values


oversampler = RandomOverSampler(random_state=42)
X_resampled, y_resampled = oversampler.fit_resample(X, y)


X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))


model_lstm_gru = Sequential()
model_lstm_gru.add(LSTM(50, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
model_lstm_gru.add(Bidirectional(GRU(50, return_sequences=False)))
model_lstm_gru.add(Dense(1, activation='sigmoid'))
model_lstm_gru.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])


model_lstm_gru.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)


y_pred_lstm_gru = model_lstm_gru.predict(X_test)
y_pred_lstm_gru = (y_pred_lstm_gru > 0.5).astype(int)


print("Classification Report for LSTM-GRU Model:")
print(classification_report(y_test, y_pred_lstm_gru))


In [ ]:
import numpy as np

# Function to accept input values and make a prediction
def predict_parkinsons():
    # Define the input values based on the features (skip 'name' and 'status')
    print("Enter the following values:")
    MDVP_Fo = float(input("MDVP:Fo(Hz): "))
    MDVP_Fhi = float(input("MDVP:Fhi(Hz): "))
    MDVP_Flo = float(input("MDVP:Flo(Hz): "))
    MDVP_Jitter_percent = float(input("MDVP:Jitter(%): "))
    MDVP_Jitter_Abs = float(input("MDVP:Jitter(Abs): "))
    MDVP_RAP = float(input("MDVP:RAP: "))
    MDVP_PPQ = float(input("MDVP:PPQ: "))
    Jitter_DDP = float(input("Jitter:DDP: "))
    MDVP_Shimmer = float(input("MDVP:Shimmer: "))
    MDVP_Shimmer_dB = float(input("MDVP:Shimmer(dB): "))
    Shimmer_APQ3 = float(input("Shimmer:APQ3: "))
    Shimmer_APQ5 = float(input("Shimmer:APQ5: "))
    MDVP_APQ = float(input("MDVP:APQ: "))
    Shimmer_DDA = float(input("Shimmer:DDA: "))
    NHR = float(input("NHR: "))
    HNR = float(input("HNR: "))
    RPDE = float(input("RPDE: "))
    DFA = float(input("DFA: "))
    spread1 = float(input("spread1: "))
    spread2 = float(input("spread2: "))
    D2 = float(input("D2: "))
    PPE = float(input("PPE: "))

    # Put all input values in a NumPy array
    input_data = np.array([[MDVP_Fo, MDVP_Fhi, MDVP_Flo, MDVP_Jitter_percent, MDVP_Jitter_Abs, MDVP_RAP, MDVP_PPQ,
                            Jitter_DDP, MDVP_Shimmer, MDVP_Shimmer_dB, Shimmer_APQ3, Shimmer_APQ5, MDVP_APQ,
                            Shimmer_DDA, NHR, HNR, RPDE, DFA, spread1, spread2, D2, PPE]])

    # Scale the input data using the same scaler used during training
    input_data_scaled = scaler.transform(input_data)

    # Reshape the input data to match the shape required by the model
    input_data_scaled = input_data_scaled.reshape((input_data_scaled.shape[0], 1, input_data_scaled.shape[1]))

    # Make prediction
    prediction = model_lstm_gru.predict(input_data_scaled)
    prediction = (prediction > 0.5).astype(int)  # Convert probabilities to binary (0 or 1)

    if prediction == 1:
        print("Prediction: The person has Parkinson's disease.")
    else:
        print("Prediction: The person does not have Parkinson's disease.")

# Call the function to make a prediction
predict_parkinsons()
